# Laboratorio 5: Minería de Textos y Análisis de Sentimiento
## Parte 1: Carga de Datos y Preprocesamiento

**Objetivo:** Cargar los datos de tweets sobre desastres y limpiarlos para análisis posterior.

---

## 1. Importar Librerías

In [23]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
import warnings
warnings.filterwarnings('ignore')

# Descargar recursos de NLTK la primera vez
import nltk
nltk.download('stopwords', quiet=True)

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Cargar los Datos

**¿Por qué?** Necesitamos traer el CSV a Jupyter como un DataFrame (tabla de datos).

In [13]:
# Cargar el archivo CSV
df = pd.read_csv('train.csv')

print(f"✓ Dataset cargado")
print(f"  Dimensiones: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"\nPrimeras 5 filas:")
df.head()

✓ Dataset cargado
  Dimensiones: 7613 filas, 5 columnas

Primeras 5 filas:


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## 3. Exploración Inicial de los Datos

**¿Por qué?** Entender qué datos tenemos, qué falta, qué tipos son.

In [25]:
# Ver información general

print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("ESTADÍSTICAS DESCRIPTIVAS")

df.describe()

INFORMACIÓN GENERAL DEL DATASET
<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            7613 non-null   int64
 1   keyword       7552 non-null   str  
 2   location      5080 non-null   str  
 3   text          7613 non-null   str  
 4   target        7613 non-null   int64
 5   text_cleaned  7613 non-null   str  
dtypes: int64(2), str(4)
memory usage: 357.0 KB

ESTADÍSTICAS DESCRIPTIVAS


,id,target
count,7613.000000,7613.00000
mean,5441.934848,0.42966
std,3137.116090,0.49506
min,1.000000,0.00000
25%,2734.000000,0.00000
50%,5408.000000,0.00000
75%,8146.000000,1.00000
max,10873.000000,1.00000


In [26]:
# Valores nulos (datos faltantes)

print("VALORES NULOS POR COLUMNA")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Nulos': missing.values,
    'Porcentaje': missing_pct.values
})
print(missing_df)

VALORES NULOS POR COLUMNA
        Columna  Nulos  Porcentaje
0            id      0    0.000000
1       keyword     61    0.801261
2      location   2533   33.272035
3          text      0    0.000000
4        target      0    0.000000
5  text_cleaned      0    0.000000


In [30]:
# Distribución de la variable objetivo (target)

print("DISTRIBUCIÓN DE LA VARIABLE TARGET")
print("=" * 60)
print(f"Desastres reales (1): {(df['target'] == 1).sum()} tweets ({(df['target'] == 1).sum()/len(df)*100:.2f}%)")
print(f"No desastres (0):     {(df['target'] == 0).sum()} tweets ({(df['target'] == 0).sum()/len(df)*100:.2f}%)")

# Ver algunos ejemplos

print("EJEMPLO DE TWEETS")

print("\n Ejemplo de DESASTRE (target=1):")
print(df[df['target']==1]['text'].iloc[0])
print("\n Ejemplo de NO DESASTRE (target=0):")
print(df[df['target']==0]['text'].iloc[0])

DISTRIBUCIÓN DE LA VARIABLE TARGET
Desastres reales (1): 3271 tweets (42.97%)
No desastres (0):     4342 tweets (57.03%)
EJEMPLO DE TWEETS

 Ejemplo de DESASTRE (target=1):
Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all

 Ejemplo de NO DESASTRE (target=0):
What's up man?


## 4. Preprocesamiento de Texto

**¿Por qué?** Los tweets tienen "ruido" que no ayuda a clasificar:
- URLs, caracteres especiales, emoticones
- Stopwords ("el", "la", "de") que aparecen en todo
- Variaciones de mayúsculas que significan lo mismo

Limpiar los datos mejora la calidad del análisis.

### Paso 1: Crear función de limpieza

In [17]:
# Obtener stopwords en inglés (el dataset está en inglés)
stop_words = set(stopwords.words('english'))

def limpiar_tweet(texto):
    """
    Limpia un tweet aplicando varias transformaciones.
    
    Pasos:
    1. Convertir a minúsculas
    2. Quitar URLs
    3. Quitar menciones (@usuario) y hashtags (#tema)
    4. Quitar emoticones y caracteres especiales
    5. Quitar números
    6. Quitar puntuación
    7. Quitar espacios extra
    8. Quitar stopwords (palabras muy comunes)
    """
    
    # 1. Convertir a minúsculas
    texto = texto.lower()
    
    # 2. Quitar URLs (http://... o https://...)
    texto = re.sub(r'http\S+|www\S+', '', texto)
    
    # 3. Quitar menciones (@usuario)
    texto = re.sub(r'@\w+', '', texto)
    
    # 4. Quitar hashtags pero GUARDAR la palabra (ej: #fire -> fire)
    texto = re.sub(r'#', '', texto)
    
    # 5. Quitar números
    texto = re.sub(r'\d+', '', texto)
    
    # 6. Quitar puntuación (excepto algunos caracteres que podrían ser útiles)
    texto = re.sub(r'[^a-z\s]', '', texto)
    
    # 7. Quitar espacios múltiples
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    # 8. Quitar stopwords
    palabras = texto.split()
    palabras = [p for p in palabras if p not in stop_words]
    texto = ' '.join(palabras)
    
    return texto

print("✓ Función de limpieza creada")

✓ Función de limpieza creada


### Paso 2: Probar la función con ejemplos

In [31]:
# Probar con ejemplos ANTES y DESPUÉS

print("EJEMPLOS DE LIMPIEZA")


ejemplos = df['text'].head(3).tolist()

for i, tweet in enumerate(ejemplos, 1):
    limpio = limpiar_tweet(tweet)
    print(f"\n EJEMPLO {i}")
    print(f"ANTES:  {tweet}")
    print(f"DESPUÉS: {limpio}")
    print("-" * 80)

EJEMPLOS DE LIMPIEZA

 EJEMPLO 1
ANTES:  Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all
DESPUÉS: deeds reason earthquake may allah forgive us
--------------------------------------------------------------------------------

 EJEMPLO 2
ANTES:  Forest fire near La Ronge Sask. Canada
DESPUÉS: forest fire near la ronge sask canada
--------------------------------------------------------------------------------

 EJEMPLO 3
ANTES:  All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected
DESPUÉS: residents asked shelter place notified officers evacuation shelter place orders expected
--------------------------------------------------------------------------------


### Paso 3: Aplicar limpieza a TODO el dataset

In [19]:
# Crear nueva columna con textos limpios
print("Limpiando todos los tweets... (esto puede tomar 10-20 segundos)\n")

df['text_cleaned'] = df['text'].apply(limpiar_tweet)

print("✓ Todos los tweets limpiados")
print(f"\nMuestra de datos limpios:")
print(df[['text', 'text_cleaned']].head(10))

Limpiando todos los tweets... (esto puede tomar 10-20 segundos)

✓ Todos los tweets limpiados

Muestra de datos limpios:
                                                text  \
0  Our Deeds are the Reason of this #earthquake M...   
1             Forest fire near La Ronge Sask. Canada   
2  All residents asked to 'shelter in place' are ...   
3  13,000 people receive #wildfires evacuation or...   
4  Just got sent this photo from Ruby #Alaska as ...   
5  #RockyFire Update => California Hwy. 20 closed...   
6  #flood #disaster Heavy rain causes flash flood...   
7  I'm on top of the hill and I can see a fire in...   
8  There's an emergency evacuation happening now ...   
9  I'm afraid that the tornado is coming to our a...   

                                        text_cleaned  
0       deeds reason earthquake may allah forgive us  
1              forest fire near la ronge sask canada  
2  residents asked shelter place notified officer...  
3  people receive wildfires evacuation ord

### Paso 4: Verificar la limpieza

In [32]:
# Estadísticas de limpieza

print("ESTADÍSTICAS DE LIMPIEZA")


# Largo promedio de tweets
print(f"\nLargo promedio de texto ORIGINAL: {df['text'].str.len().mean():.2f} caracteres")
print(f"Largo promedio de texto LIMPIO:   {df['text_cleaned'].str.len().mean():.2f} caracteres")

# Tweets vacíos después de limpiar
vacios = (df['text_cleaned'].str.len() == 0).sum()
print(f"\nTweets que quedaron vacíos: {vacios} ({vacios/len(df)*100:.2f}%)")

# Si hay tweets vacíos, quitarlos
if vacios > 0:
    print(f"\nEliminando {vacios} tweets vacíos...")
    df = df[df['text_cleaned'].str.len() > 0].reset_index(drop=True)
    print(f"✓ Dataset actualizado: {len(df)} tweets")

ESTADÍSTICAS DE LIMPIEZA

Largo promedio de texto ORIGINAL: 101.04 caracteres
Largo promedio de texto LIMPIO:   59.91 caracteres

Tweets que quedaron vacíos: 0 (0.00%)


## 5. Resumen del Preprocesamiento

**¿Qué hicimos?**

In [34]:

print("RESUMEN DEL PREPROCESAMIENTO")


resumen = """
✓ CARGAS Y EXPLORACIÓN:
  • Dataset: 10,500+ tweets sobre desastres naturales
  • Columnas: id, keyword, location, text, target
  • Target: 1=Desastre real | 0=No desastre
  • Balanceo: ~43% desastres, ~57% no desastres

✓ LIMPIEZA APLICADA:
  1. Convertir a minúsculas ("FIRE" → "fire")
  2. Quitar URLs (no aportan al significado)
  3. Quitar menciones (@usuario) - no relevante
  4. Quitar hashtags pero guardar palabra (#fire → fire)
  5. Quitar números (13, 911, etc.)
  6. Quitar puntuación y caracteres especiales
  7. Quitar espacios múltiples
  8. Quitar stopwords (el, la, de, y, etc.)

✓ RESULTADO:
  • Textos más cortos y enfocados
  • Solo palabras que aportan significado
  • Listos para análisis exploratorio

✓ COLUMNA NUEVA:
  • 'text_cleaned': tweets procesados listos para análisis
"""

print(resumen)


RESUMEN DEL PREPROCESAMIENTO

✓ CARGAS Y EXPLORACIÓN:
  • Dataset: 10,500+ tweets sobre desastres naturales
  • Columnas: id, keyword, location, text, target
  • Target: 1=Desastre real | 0=No desastre
  • Balanceo: ~43% desastres, ~57% no desastres

✓ LIMPIEZA APLICADA:
  1. Convertir a minúsculas ("FIRE" → "fire")
  2. Quitar URLs (no aportan al significado)
  3. Quitar menciones (@usuario) - no relevante
  4. Quitar hashtags pero guardar palabra (#fire → fire)
  5. Quitar números (13, 911, etc.)
  6. Quitar puntuación y caracteres especiales
  7. Quitar espacios múltiples
  8. Quitar stopwords (el, la, de, y, etc.)

✓ RESULTADO:
  • Textos más cortos y enfocados
  • Solo palabras que aportan significado
  • Listos para análisis exploratorio

✓ COLUMNA NUEVA:
  • 'text_cleaned': tweets procesados listos para análisis



## 6. Guardar Dataset Limpio

**¿Por qué?** Para usarlo en la siguiente parte sin tener que limpiar de nuevo.

In [22]:
# Guardar el dataset limpio
df.to_csv('train_cleaned.csv', index=False)

print("✓ Dataset limpio guardado como 'train_cleaned.csv'")
print(f"\nDataset final:")
print(df.head())

✓ Dataset limpio guardado como 'train_cleaned.csv'

Dataset final:
   id keyword location                                               text  \
0   1     NaN      NaN  Our Deeds are the Reason of this #earthquake M...   
1   4     NaN      NaN             Forest fire near La Ronge Sask. Canada   
2   5     NaN      NaN  All residents asked to 'shelter in place' are ...   
3   6     NaN      NaN  13,000 people receive #wildfires evacuation or...   
4   7     NaN      NaN  Just got sent this photo from Ruby #Alaska as ...   

   target                                       text_cleaned  
0       1       deeds reason earthquake may allah forgive us  
1       1              forest fire near la ronge sask canada  
2       1  residents asked shelter place notified officer...  
3       1  people receive wildfires evacuation orders cal...  
4       1  got sent photo ruby alaska smoke wildfires pou...  
